In [2]:
"""
23 Feb 2026
stability_analysis.py
=====================
Numerical verification of Jacobian eigenvalue signs for the extended
Abrams-Strogatz model with demographic growth.

THREE SCENARIOS (mirrors language_dynamics_extended_v5.py):
  Scenario A  — single global fit, whole period 1895-2020 (a=1.25)
  Scenario B  — dual-regime:
    B1: Consensus  1895-1970  (a=1.478, analytically derived)
    B2: Coexistence 1970-2020 (a=0.655)

For each scenario the script:
  1. Finds x_o* at each census / diagnostic year
  2. Computes J00 (x_o-direction eigenvalue) and J11 (I-direction eigenvalue)
  3. Reports the dominance ratio |C00|/|f'|
  4. Classifies each fixed point (saddle / stable node / unstable)

Author: Riccardo Del Gratta
Date: 2025
"""

import numpy as np
from scipy.optimize import brentq

# ============================================================================
# MODEL PARAMETERS
# ============================================================================

N0_indigenous = 1012848
K_indigenous  = 12918933
r_indigenous  = 0.022

N0_spanish = 3117878
K_spanish  = 165995301
r_spanish  = 0.037

A_p   = 5.47
nu    = 0.271
p_max = 0.97

base_year = 1895

# ---------------------------------------------------------------------------
# THREE SCENARIOS
# ---------------------------------------------------------------------------
# Scenario A  — single global fit, 1895-2020
S_O_CONS_ALL = 0.0349;  S_L_CONS_ALL = 0.0055;  A_CONS_ALL = 1.25

# Scenario B  — dual-regime
#   B1: Consensus  1895-1970  (analytically derived)
S_O_CONS = 0.0454;  S_L_CONS = 0.00839;  A_CONS = 1.478
#   B2: Coexistence 1970-2020
S_O_COEX = 0.047;   S_L_COEX = 0.015;   A_COEX = 0.6553

T_REGIME_CHANGE = 75.0   # t = year 1970

# Legacy aliases (backward-compatibility)
s_o_cons = S_O_CONS_ALL;  s_l_cons = S_L_CONS_ALL;  a_cons = A_CONS_ALL
s_o_coex = S_O_COEX;      s_l_coex = S_L_COEX;      a_coex = A_COEX

SCENARIOS = {
    'A': {
        'label': 'A — Single-fit (a=1.25, 1895-2020)',
        'cons': dict(s_o=S_O_CONS_ALL, s_l=S_L_CONS_ALL, a=A_CONS_ALL),
        'coex': dict(s_o=S_O_CONS_ALL, s_l=S_L_CONS_ALL, a=A_CONS_ALL),
        'dual': False,
    },
    'B': {
        'label': 'B — Dual-regime (B1: a=1.478 / B2: a=0.655)',
        'cons': dict(s_o=S_O_CONS, s_l=S_L_CONS, a=A_CONS),
        'coex': dict(s_o=S_O_COEX, s_l=S_L_COEX, a=A_COEX),
        'dual': True,
    },
}


def scenario_params(t, scenario='A'):
    """
    Return (s_o, s_l, a, sub-label) for the given scenario at time t.

    Parameters
    ----------
    t        : float  years since base_year (1895)
    scenario : str    'A' or 'B'
    """
    sc = SCENARIOS[scenario]
    if t < T_REGIME_CHANGE:
        p   = sc['cons']
        lbl = ('A (1895-1970)' if not sc['dual']
                else 'B1-Consensus  (a=1.478)')
    else:
        p   = sc['coex']
        lbl = ('A (1970-2020)' if not sc['dual']
                else 'B2-Coexistence (a=0.655)')
    return p['s_o'], p['s_l'], p['a'], lbl


# ============================================================================
# CORE FUNCTIONS
# ============================================================================

def logistic(t, K, r, N0):
    return K / (1 + ((K - N0) / N0) * np.exp(-r * t))


def p_o_func(m):
    return p_max / (1 + A_p * np.exp(-nu * m))


def f_func(x_o, s_o, s_l, a):
    if x_o <= 0 or x_o >= 1:
        return 0.0
    return s_o * (x_o**a) * (1 - x_o) - s_l * ((1 - x_o)**a) * x_o


def g_func(x_o, p, m_val, I):
    bracket = p - x_o - p * (1 - p / p_max) * nu * m_val
    return bracket * r_indigenous * (1 - I / K_indigenous)


def rhs(x_o, I, S, s_o, s_l, a):
    m_val = S / I if I > 0 else np.inf
    p = p_o_func(m_val)
    return f_func(x_o, s_o, s_l, a) + g_func(x_o, p, m_val, I)


# ============================================================================
# STABILITY ANALYSIS — single scenario, given parameter set
# ============================================================================

def analyze_stability(regime_name, s_o, s_l, a, years):
    """
    Print Jacobian diagnostics for a single (s_o, s_l, a) parameter set
    at the specified years.

    Columns: Year | I/K_I | x_o* | J00 | J11 | |C00|/|f'| | classification
    """
    C_WIDTH = 105
    print("\n" + "=" * C_WIDTH)
    print(f"  {regime_name}   (a={a}, s_o={s_o}, s_l={s_l})")
    print("=" * C_WIDTH)
    ratio_hdr = "|C00|/|f'|"
    hdr = (f"{'Year':>4}  {'I/K_I':>6}  {'x_o*':>7}  "
           f"{'J00':>9}  {'J11':>9}  {ratio_hdr:>11}  "
           f"{'sgn(J00)':>8}  {'sgn(J11)':>8}  Classification")
    print(hdr)
    print("-" * C_WIDTH)

    for year in years:
        t   = year - base_year
        I_t = logistic(t, K_indigenous, r_indigenous, N0_indigenous)
        S_t = logistic(t, K_spanish,    r_spanish,    N0_spanish)

        # --- find fixed point ---
        xs   = np.linspace(0.01, 0.99, 3000)
        vals = [rhs(x, I_t, S_t, s_o, s_l, a) for x in xs]
        scs  = np.where(np.diff(np.sign(vals)))[0]
        if len(scs) == 0:
            print(f"{year:>4}  {I_t/K_indigenous:>6.3f}  --- no interior FP ---")
            continue

        x_star = brentq(lambda x: rhs(x, I_t, S_t, s_o, s_l, a),
                        xs[scs[0]], xs[scs[0]+1], xtol=1e-10)

        # --- J00 (centred finite diff) ---
        eps = 1e-6
        xp  = min(x_star + eps, 0.999)
        xm  = max(x_star - eps, 0.001)
        J00 = (rhs(xp, I_t, S_t, s_o, s_l, a)
               - rhs(xm, I_t, S_t, s_o, s_l, a)) / (xp - xm)

        # --- J11 (analytic) ---
        J11 = r_indigenous * (1 - 2 * I_t / K_indigenous)

        # --- dominance ratio |C00|/|f'| ---
        fp_num = (f_func(min(x_star+eps,0.9999), s_o, s_l, a)
                 -f_func(max(x_star-eps,0.0001), s_o, s_l, a)) / (2*eps)
        c00    = -r_indigenous * (1 - I_t / K_indigenous)
        ratio  = abs(c00) / abs(fp_num) if abs(fp_num) > 1e-15 else float('inf')

        # --- classification ---
        if   J00 < 0 and J11 < 0: cls = "Stable node"
        elif J00 < 0 and J11 > 0: cls = "Saddle (x stab, I unstab)"
        elif J00 > 0 and J11 < 0: cls = "Saddle (x unstab, I stab)"
        else:                      cls = "Unstable node"

        s00 = "neg" if J00 < 0 else "pos"
        s11 = "neg" if J11 < 0 else "pos"

        print(f"{year:>4}  {I_t/K_indigenous:>6.3f}  {x_star:>7.4f}  "
              f"{J00:>+9.5f}  {J11:>+9.5f}  {ratio:>11.3f}  "
              f"{s00:>8}  {s11:>8}  {cls}")


# ============================================================================
# FULL THREE-SCENARIO COMPARISON TABLE
# ============================================================================

def print_three_scenario_table(years):
    """
    Print side-by-side: Scenario A | Scenario B1 (pre-1970) or B2 (post-1970)
    with J00, J11, |C00|/|f'|, and classification for each.
    Mirrors the comparison logic of language_dynamics_extended_v5.py.
    """
    C_WIDTH = 140
    print("\n" + "=" * C_WIDTH)
    print("  THREE-SCENARIO COMPARISON  —  J00, J11, dominance ratio at each year")
    print("  Scenario A (a=1.25 throughout)  |  Scenario B (B1: a=1.478 / B2: a=0.655)")
    print("=" * C_WIDTH)
    print(f"{'Year':>4}  {'Regime_A':>14}  "
          f"{'xo*(A)':>7}  {'J00(A)':>9}  {'J11(A)':>9}  {'ratio(A)':>8}  {'cls(A)':>26}  ||  "
          f"{'Regime_B':>22}  "
          f"{'xo*(B)':>7}  {'J00(B)':>9}  {'J11(B)':>9}  {'ratio(B)':>8}  {'cls(B)':>26}")
    print("-" * C_WIDTH)

    rms_sq = {'A': [], 'B': []}

    for year in years:
        t   = year - base_year
        I_t = logistic(t, K_indigenous, r_indigenous, N0_indigenous)
        S_t = logistic(t, K_spanish,    r_spanish,    N0_spanish)

        row = {}
        for sc in ('A', 'B'):
            s_o, s_l, a, lbl = scenario_params(t, sc)

            xs   = np.linspace(0.01, 0.99, 3000)
            vals = [rhs(x, I_t, S_t, s_o, s_l, a) for x in xs]
            scs  = np.where(np.diff(np.sign(vals)))[0]
            if len(scs) == 0:
                row[sc] = dict(lbl=lbl, xr=float('nan'), J00=float('nan'),
                               J11=float('nan'), ratio=float('nan'), cls='no FP')
                continue

            xr = brentq(lambda x: rhs(x, I_t, S_t, s_o, s_l, a),
                        xs[scs[0]], xs[scs[0]+1], xtol=1e-10)

            eps = 1e-6
            xp  = min(xr+eps, 0.999);  xm = max(xr-eps, 0.001)
            J00 = (rhs(xp, I_t, S_t, s_o, s_l, a)
                  -rhs(xm, I_t, S_t, s_o, s_l, a)) / (xp - xm)
            J11 = r_indigenous * (1 - 2*I_t/K_indigenous)

            fp_num = (f_func(min(xr+eps,0.9999),s_o,s_l,a)
                     -f_func(max(xr-eps,0.0001),s_o,s_l,a)) / (2*eps)
            c00    = -r_indigenous * (1 - I_t/K_indigenous)
            ratio  = abs(c00)/abs(fp_num) if abs(fp_num)>1e-15 else float('inf')

            if   J00 < 0 and J11 < 0: cls = "Stable node"
            elif J00 < 0 and J11 > 0: cls = "Saddle (x stb, I unst)"
            elif J00 > 0 and J11 < 0: cls = "Saddle (x unst, I stb)"
            else:                      cls = "Unstable node"

            row[sc] = dict(lbl=lbl, xr=xr, J00=J00, J11=J11, ratio=ratio, cls=cls)

        A = row['A'];  B = row['B']
        flag = " ←" if year in (1960, 1970) else ""
        print(f"{year:>4}  {A['lbl']:>14}  "
              f"{A['xr']:>7.4f}  {A['J00']:>+9.5f}  {A['J11']:>+9.5f}  "
              f"{A['ratio']:>8.3f}  {A['cls']:>26}  ||  "
              f"{B['lbl']:>22}  "
              f"{B['xr']:>7.4f}  {B['J00']:>+9.5f}  {B['J11']:>+9.5f}  "
              f"{B['ratio']:>8.3f}  {B['cls']:>26}{flag}")

    print()


# ============================================================================
# RMS RESIDUALS (if census data available)
# ============================================================================

CENSUS_XO_OBS = {
    1895: 0.2886, 1900: 0.2949, 1910: 0.2951, 1930: 0.4754,
    1940: 0.5032, 1950: 0.6752, 1960: 0.6353, 1970: 0.7235,
    1980: 0.7591, 1990: 0.8352, 1995: 0.8519, 2000: 0.8310,
    2005: 0.8772, 2010: 0.8479, 2020: 0.8895,
}


def print_rms_table(census_years):
    """
    Compute and print RMS residuals for Scenarios A and B (B1+B2 stitched),
    with sub-period breakdown (pre-1970 and post-1970) matching
    language_dynamics_extended_v5.py output.
    """
    sq  = {'A': [], 'B': [], 'A_pre': [], 'A_post': [], 'B1': [], 'B2': []}

    for year in census_years:
        if year not in CENSUS_XO_OBS:
            continue
        xo_obs = CENSUS_XO_OBS[year]
        t      = year - base_year
        I_t    = logistic(t, K_indigenous, r_indigenous, N0_indigenous)
        S_t    = logistic(t, K_spanish,    r_spanish,    N0_spanish)

        for sc in ('A', 'B'):
            s_o, s_l, a, _ = scenario_params(t, sc)
            xs   = np.linspace(0.01, 0.99, 3000)
            vals = [rhs(x, I_t, S_t, s_o, s_l, a) for x in xs]
            scs  = np.where(np.diff(np.sign(vals)))[0]
            if len(scs) == 0:
                continue
            xr  = brentq(lambda x: rhs(x, I_t, S_t, s_o, s_l, a),
                         xs[scs[0]], xs[scs[0]+1], xtol=1e-10)
            res = xr - xo_obs
            sq[sc].append(res**2)
            if sc == 'A':
                (sq['A_pre'] if year <= 1970 else sq['A_post']).append(res**2)
            else:
                (sq['B1'] if year <= 1970 else sq['B2']).append(res**2)

    print("\n  RMS residuals:")
    print(f"    Scenario A (full):   RMS = {np.sqrt(np.mean(sq['A'])):.5f}")
    print(f"    Scenario B (full):   RMS = {np.sqrt(np.mean(sq['B'])):.5f}")
    print(f"    Scenario A 1895-1970: RMS = {np.sqrt(np.mean(sq['A_pre'])):.5f}")
    print(f"    Scenario A 1970-2020: RMS = {np.sqrt(np.mean(sq['A_post'])):.5f}")
    print(f"    Scenario B1 1895-1970: RMS = {np.sqrt(np.mean(sq['B1'])):.5f}")
    print(f"    Scenario B2 1970-2020: RMS = {np.sqrt(np.mean(sq['B2'])):.5f}")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":

    print("\n" + "#" * 105)
    print("# STABILITY ANALYSIS — Extended Abrams-Strogatz model")
    print("# Mexican bilingual dynamics (Spanish/Indigenous) 1895-2020")
    print("# THREE SCENARIOS: A (single-fit) | B1 (Consensus) | B2 (Coexistence)")
    print("#" * 105)

    # ------------------------------------------------------------------
    # Scenario A — full period with single a=1.25
    # ------------------------------------------------------------------
    A_years = [1895, 1910, 1930, 1940, 1950, 1960, 1970,
               1980, 1990, 2000, 2010, 2020]
    analyze_stability(
        "SCENARIO A — Single-fit (a=1.25, 1895-2020)",
        S_O_CONS_ALL, S_L_CONS_ALL, A_CONS_ALL,
        A_years
    )
    print("""
  Key observations — Scenario A:
    J00 < 0 throughout: x_o direction stable.
    Dominant stabiliser: C00 (demographic) for 1895-1960; f' grows after 1970.
    J11 changes sign ~2007: saddle (1895-2007) → stable node (2007-2020).
""")

    # ------------------------------------------------------------------
    # Scenario B1 — Consensus 1895-1970  (a=1.478)
    # ------------------------------------------------------------------
    B1_years = [1895, 1910, 1930, 1940, 1950, 1960, 1970]
    analyze_stability(
        "SCENARIO B1 — Consensus (a=1.478, 1895-1970)",
        S_O_CONS, S_L_CONS, A_CONS,
        B1_years
    )
    print("""
  Key observations — Scenario B1 (a=1.478):
    J00 < 0 throughout B1 period: C00 is sole stabiliser (f' > 0 throughout).
    Higher a → steeper repulsion near x=0 → low attractor 1930-1940 (residual -0.32).
    All B1 points are saddles (J11 > 0 until 2007, which is after B1 ends).
""")

    # ------------------------------------------------------------------
    # Scenario B2 — Coexistence 1970-2020  (a=0.655)
    # ------------------------------------------------------------------
    B2_years = [1970, 1980, 1990, 2000, 2007, 2010, 2020]
    analyze_stability(
        "SCENARIO B2 — Coexistence (a=0.655, 1970-2020)",
        S_O_COEX, S_L_COEX, A_COEX,
        B2_years
    )
    print("""
  Key observations — Scenario B2 (a=0.655):
    J00 < 0: f' < 0 (linguistic dominant), C00 also < 0 — both stabilise.
    |f'| > |C00| throughout B2: linguistic term dominates stability.
    Transition: saddle (1970-2007) → stable node (2007-2020).
""")

    # ------------------------------------------------------------------
    # Three-scenario side-by-side comparison
    # ------------------------------------------------------------------
    all_census = sorted(CENSUS_XO_OBS.keys())
    print_three_scenario_table(all_census)

    # ------------------------------------------------------------------
    # RMS residuals
    # ------------------------------------------------------------------
    print_rms_table(all_census)

    print("\n" + "#" * 105)
    print("# SUMMARY")
    print("#   Scenario A:  J00 < 0 always; single stable attractor throughout.")
    print("#   Scenario B1: J00 < 0; C00 sole stabiliser; low attractor 1930-1940.")
    print("#   Scenario B2: J00 < 0; linguistic f' dominant stabiliser post-1970.")
    print("#   All scenarios: saddle (1895-2007) → stable node (2007-2020).")
    print("#   Dominance crossover: |C00| > |f'| early; |f'| > |C00| late.")
    print("#" * 105)


#########################################################################################################
# STABILITY ANALYSIS — Extended Abrams-Strogatz model
# Mexican bilingual dynamics (Spanish/Indigenous) 1895-2020
# THREE SCENARIOS: A (single-fit) | B1 (Consensus) | B2 (Coexistence)
#########################################################################################################

  SCENARIO A — Single-fit (a=1.25, 1895-2020)   (a=1.25, s_o=0.0349, s_l=0.0055)
Year   I/K_I     x_o*        J00        J11   |C00|/|f'|  sgn(J00)  sgn(J11)  Classification
---------------------------------------------------------------------------------------------------------
1895   0.078   0.3807   -0.01019   +0.01855        2.011       neg       pos  Saddle (x stab, I unstab)
1910   0.106   0.3632   -0.00884   +0.01734        1.817       neg       pos  Saddle (x stab, I unstab)
1930   0.155   0.3398   -0.00682   +0.01517        1.579       neg       pos  Saddle (x stab, I unstab)
1940   0.1